In [1]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

# Load and pre-process data

In [2]:
target_col = "log_average_ms"

node_cols = ["Abs", "Acos", "Add", "ai.onnx.ml::CategoryMapper", "And", "ArgMax", 
             "AveragePool", "BatchNormalization", "Cast", "Ceil", "Clip", 
             "com.microsoft::BiasGelu", "com.microsoft::DynamicQuantizeLSTM", 
             "com.microsoft::DynamicQuantizeMatMul", "com.microsoft::FastGelu", 
             "com.microsoft::FusedConv", "com.microsoft::FusedGemm", 
             "com.microsoft::FusedMatMul", "com.microsoft::MatMulIntegerToFloat", 
             "com.microsoft::QGemm", "com.microsoft::QLinearAdd", 
             "com.microsoft::QLinearAveragePool", "com.microsoft::QLinearConcat", 
             "com.microsoft::QLinearGlobalAveragePool", "com.microsoft::QLinearLeakyRelu", 
             "com.microsoft::QLinearMul", "com.microsoft::QLinearSigmoid", 
             "com.microsoft::QuickGelu", "com.microsoft::SkipLayerNormalization", 
             "Compress", "Concat", "Constant", "ConstantOfShape", "Conv", "ConvInteger", 
             "ConvTranspose", "Cos", "CumSum", "DequantizeLinear", "Div", "Dropout", 
             "DynamicQuantizeLinear", "Einsum", "Equal", "Erf", "Exp", "Expand", 
             "EyeLike", "Flatten", "Floor", "Gather", "GatherElements", "GatherND", 
             "Gelu", "Gemm", "GlobalAveragePool", "Greater", "GreaterOrEqual", "Hardmax", 
             "HardSigmoid", "HardSwish", "Identity", "If", "InstanceNormalization", 
             "LayerNormalization", "LeakyRelu", "Less", "LessOrEqual", "local::preprocess", 
             "Log", "LogSoftmax", "Loop", "LRN", "LSTM", "MatMul", "MatMulInteger", 
             "Max", "MaxPool", "Min", "Mod", "Mul", "Neg", "NonMaxSuppression", "NonZero", 
             "Not", "OneHot", "Or", "Pad", "Pow", "PRelu", "QLinearConv", "QLinearMatMul", 
             "QuantizeLinear", "Range", "Reciprocal", "ReduceMax", "ReduceMean", "ReduceMin", 
             "ReduceProd", "ReduceSum", "Relu", "Reshape", "Resize", "RoiAlign", "Round", 
             "Scan", "ScatterElements", "ScatterND", "Shape", "Sigmoid", 
             "SimplifiedLayerNormalization", "Sin", "Slice", "Softmax", "Split", "Sqrt", 
             "Squeeze", "Sub", "Sum", "Tanh", "Tile", "TopK", "Transpose", "Trilu", 
             "Unsqueeze", "Where", "Xor"]


# node_cols = ["Constant", "Reshape", "Add", "Conv",
#              "Mul", "Transpose", "Relu", "MatMul",
#              "Gemm", "Unsqueeze"]


# extra_cols = ["movement_frac_x_cores", "mb_per_core"]

feature_cols = node_cols + ["conv_flops", "matmul_flops",
                "elementwise_mb", "reduction_mb", "normalization_mb",
                "movement_mb", "memory_mb", "l1d_cache_kb",
                "l1i_cache_kb", "l2_cache_kb", "base_clock_mhz",
                "num_cores", "memory_bandwith_gbs"]

metadata_cols = ["model", "cpu_provider",
                 "machine_type", "platform", "run_id"]

In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
# load test, train, val sets

train_df = pd.read_csv("/content/drive/MyDrive/Data/train_set.csv")
val_df = pd.read_csv("/content/drive/MyDrive/Data/val_set.csv")
test_df = pd.read_csv("/content/drive/MyDrive/Data/test_set.csv")

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (100252, 154)
val: (21483, 154)
test: (21483, 154)


In [5]:
assert list(train_df.columns) == list(val_df.columns) == list(test_df.columns)

train_df.head()

,model,input_dimensions,input_dtypes,output_dimensions,output_dtypes,conv_flops,matmul_flops,elementwise_mb,reduction_mb,normalization_mb,...,base_clock_mhz,memory_bandwith_gbs,cpu_provider,machine_type,platform,repo_file,average_ms,stddev_ms,min_ms,max_ms
0,hardcorenas_d_Opset17_extended.onnx,x:1x3x224x224,x:float32,668:1x1000,668:float32,472440512,2560000,50.528229,5.204269,0.000000,...,2300.000,717,intel,xeon_plat,bluehive,hardcorenas_d_Opset17.onnx,14.069188,0.082284,13.931521,14.247209
1,vit_base_patch8_224_in21k_Opset17_disable_all....,x:1x3x224x224,x:float32,1089:1x21843,1089:float32,231211008,156097479168,1771.169711,0.000000,792.141724,...,2450.000,205,amd,epyc,gcloud,vit_base_patch8_224_in21k_Opset17.onnx,1481.456045,150.199862,1094.324125,1632.506909
2,tf_efficientnetv2_m_in21ft1k_Opset17_basic.onnx,x:1x3x384x384,x:float32,2466:1x1000,2466:float32,31470991360,2560000,1201.339462,67.736023,0.000000,...,2449.998,205,amd,epyc,gcloud,tf_efficientnetv2_m_in21ft1k_Opset17.onnx,301.127424,12.889699,286.464729,324.336829
3,shufflenet_v2_x1_5_Opset16.onnx,x:1x3x224x224,x:float32,1137:1x1000,1137:float32,266034528,2048000,8.246674,1.630875,0.000000,...,2449.998,205,amd,epyc,gcloud,shufflenet_v2_x1_5_Opset16.onnx,8.161246,0.036559,8.080030,8.197800
4,wide_resnet101_2_Opset16_timm.onnx,x:1x3x224x224,x:float32,971:1x1000,971:float32,45502005248,4096000,252.656250,4.218750,0.000000,...,2300.000,717,intel,xeon_plat,bluehive,wide_resnet101_2_Opset16_timm.onnx,38.147223,0.068437,38.035275,38.292409


In [6]:
# pre-processing function

def preprocess(df: pd.DataFrame, train=False) -> pd.DataFrame:

  #drop rows with missing features
  df = df.dropna(subset=["average_ms"])

  #remove rows with high standard deviation
  cv_limit = 0.1 # 10% cv limit
  df["cv"] = df["stddev_ms"] / df["average_ms"]

  # if train:
  df = df[df["cv"] <= cv_limit].copy()

  #add log_average_ms column
  df["log_average_ms"] = np.log(df["average_ms"])

  #remove decimals and round down
  for col in ["elementwise_mb", "reduction_mb", "normalization_mb", "movement_mb"]:
    df[col] = df[col].round(0)

  return df

In [7]:
train_df = preprocess(train_df, True)
test_df = preprocess(test_df, False)
val_df = preprocess(val_df, False)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (95814, 156)
val: (20494, 156)
test: (20528, 156)


# Model Training

In [8]:
from sklearn.model_selection import ParameterSampler
from sklearn.ensemble import ExtraTreesRegressor


In [9]:
X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_val = val_df[feature_cols]
y_val = val_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

In [10]:
def rmse_log(y_true, y_pred):
  return np.sqrt(mean_squared_error(y_true, y_pred))

In [11]:
def train_extra_trees(params, X_train, y_train, X_val, y_val):
    model = ExtraTreesRegressor(
        random_state=67,
        n_jobs=-1,
        **params,
    )

    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)
    score = rmse_log(y_val, val_pred)

    return model, score

In [15]:
param_dist = {
    "n_estimators": [300, 600, 900],
    "max_depth": [None, 20, 40],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", 0.5, 1.0],
    "bootstrap": [False],
}

In [16]:
sampled_params = list(ParameterSampler(
    param_dist,
    n_iter=10,
    random_state=67,
))

In [17]:
results = []

best_model = None
best_score = float("inf")
best_params = None

for i, params in enumerate(sampled_params, start=1):
    model, val_rmse = train_extra_trees(params, X_train, y_train, X_val, y_val)

    row = {
        "run": i,
        "val_rmse_log": val_rmse,
        **params,
    }

    results.append(row)

    if val_rmse < best_score:
        best_score = val_rmse
        best_model = model
        best_params = params

    print(f"{i:02d} | val RMSE log = {val_rmse:.5f} | best = {best_score:.5f}")

01 | val RMSE log = 0.28071 | best = 0.28071
02 | val RMSE log = 0.20261 | best = 0.20261
03 | val RMSE log = 0.29044 | best = 0.20261
04 | val RMSE log = 0.22767 | best = 0.20261
05 | val RMSE log = 0.72462 | best = 0.20261
06 | val RMSE log = 0.22505 | best = 0.20261
07 | val RMSE log = 0.20251 | best = 0.20251
08 | val RMSE log = 0.21601 | best = 0.20251
09 | val RMSE log = 0.20774 | best = 0.20251
10 | val RMSE log = 0.19968 | best = 0.19968


In [18]:
tuning_results = pd.DataFrame(results).sort_values("val_rmse_log")
tuning_results.head(10)

,run,val_rmse_log,n_estimators,min_samples_split,min_samples_leaf,max_features,max_depth,bootstrap
9,10,0.199679,600,5,1,1.0,40.0,False
6,7,0.202509,900,2,2,0.5,40.0,False
1,2,0.202611,300,2,2,0.5,NaN,False
8,9,0.207739,900,2,5,1.0,NaN,False
7,8,0.216011,900,5,5,0.5,40.0,False
5,6,0.225051,900,5,1,1.0,20.0,False
3,4,0.227674,600,5,2,sqrt,NaN,False
0,1,0.280711,900,5,1,0.5,20.0,False
2,3,0.290436,600,5,5,0.5,20.0,False
4,5,0.724616,900,2,1,sqrt,20.0,False


# Evaluation

In [19]:
final_model = best_model

test_pred = final_model.predict(X_test)
test_rmse_log = np.sqrt(mean_squared_error(y_test, test_pred))

print("Test RMSE: ", test_rmse_log)

Test RMSE:  0.18887127074258495


In [20]:
def latency_metrics_from_log(y_log_true, y_log_pred):
    true_ms = np.exp(y_log_true)
    pred_ms = np.exp(y_log_pred)

    error_ms = pred_ms - true_ms
    rel_err = np.abs(error_ms) / true_ms
    ratio_err = np.maximum(pred_ms / true_ms, true_ms / pred_ms)

    rmse_log = np.sqrt(mean_squared_error(y_log_true, y_log_pred))
    rmse_ms = np.sqrt(np.mean(error_ms ** 2))
    rmse_percent = np.sqrt(np.mean(((error_ms / true_ms) * 100) ** 2))

    return {
        "rmse_log": rmse_log,
        "rmse_ms": rmse_ms,
        "rmse_percent": rmse_percent,

        "median_relative_error": np.median(rel_err),
        "p90_relative_error": np.percentile(rel_err, 90),
        "p95_relative_error": np.percentile(rel_err, 95),

        "median_percent_error": np.median(rel_err) * 100,
        "p90_percent_error": np.percentile(rel_err, 90) * 100,
        "p95_percent_error": np.percentile(rel_err, 95) * 100,

        "median_ratio_error": np.median(ratio_err),
        "p90_ratio_error": np.percentile(ratio_err, 90),

        "within_10pct": np.mean(rel_err <= 0.10),
        "within_25pct": np.mean(rel_err <= 0.25),
        "within_50pct": np.mean(rel_err <= 0.50),
        "within_2x": np.mean(ratio_err <= 2.0),
    }

In [21]:
val_metrics = latency_metrics_from_log(y_val, final_model.predict(X_val))
test_metrics = latency_metrics_from_log(y_test, final_model.predict(X_test))

pd.DataFrame([val_metrics, test_metrics], index=["val", "test"])

,rmse_log,rmse_ms,rmse_percent,median_relative_error,p90_relative_error,p95_relative_error,median_percent_error,p90_percent_error,p95_percent_error,median_ratio_error,p90_ratio_error,within_10pct,within_25pct,within_50pct,within_2x
val,0.199679,138.184445,29.384391,0.019047,0.209646,0.375017,1.904653,20.964612,37.501668,1.019207,1.235188,0.795111,0.920757,0.965746,0.982922
test,0.188871,109.896726,23.538942,0.018893,0.207549,0.373858,1.889280,20.754937,37.385823,1.019076,1.233467,0.799347,0.920012,0.966923,0.983827


# Save Model

In [22]:
import joblib
from pathlib import Path

In [24]:
bundle = {
    "model": best_model,
    "feature_cols": feature_cols,
    "target_col": "log_average_ms",
}

model_dir = Path("/content/drive/MyDrive/Models")
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(bundle, model_dir / "8.joblib")

['/content/drive/MyDrive/Models/8.joblib']

# Analysis

In [124]:
test_results = test_df.copy()

test_results["pred_log_latency"] = test_pred
test_results["true_log_latency"] = y_test

test_results["pred_latency_ms"] = np.exp(test_results["pred_log_latency"])
test_results["true_latency_ms"] = np.exp(test_results["true_log_latency"])

test_results["relative_error"] = (
    np.abs(test_results["pred_latency_ms"] - test_results["true_latency_ms"])
    / test_results["true_latency_ms"]
)

In [123]:
def group_latency_metrics(g):
    y_true = g["true_log_latency"].to_numpy()
    y_pred = g["pred_log_latency"].to_numpy()

    true_ms = np.exp(y_true)
    pred_ms = np.exp(y_pred)

    rel_err = np.abs(pred_ms - true_ms) / true_ms

    return pd.Series({
        "count": len(g),
        "rmse_log": np.sqrt(mean_squared_error(y_true, y_pred)),
        "within_10pct": np.mean(rel_err <= 0.10),
        "within_25pct": np.mean(rel_err <= 0.25),
        "within_50pct": np.mean(rel_err <= 0.50),
    })

In [125]:
test_results["abs_log_error"] = np.abs(
    test_results["pred_log_latency"] - test_results["true_log_latency"]
)

test_results.groupby(["cpu_provider", "platform", "num_cores"])["relative_error"].agg(
    count="count",
    median="median",
    p90=lambda x: x.quantile(0.9),
    within_10pct=lambda x: (x <= 0.10).mean(),
    within_25pct=lambda x: (x <= 0.25).mean(),
).reset_index()

,cpu_provider,platform,num_cores,count,median,p90,within_10pct,within_25pct
0,amd,gcloud,1,2636,0.014014,0.107583,0.889605,0.971927
1,amd,gcloud,2,4015,0.044622,0.184570,0.746700,0.948443
2,amd,gcloud,4,2737,0.054011,0.254592,0.678845,0.896967
3,intel,bluehive,1,1347,0.020061,0.202241,0.810690,0.933185
4,intel,bluehive,2,2686,0.010773,0.174723,0.846612,0.925540
5,intel,bluehive,4,3999,0.012097,0.284413,0.785446,0.889722
6,intel,bluehive,8,2712,0.033049,0.506414,0.656711,0.785398
7,intel,gcloud,6,1351,0.034427,0.300699,0.736491,0.874167


In [126]:
test_results.groupby(["cpu_provider", "platform", "num_cores"])["cv"].agg(
    count="count",
    median="median",
    p90=lambda x: x.quantile(0.9),
).reset_index()

,cpu_provider,platform,num_cores,count,median,p90
0,amd,gcloud,1,2636,0.008788,0.027356
1,amd,gcloud,2,4015,0.031481,0.086614
2,amd,gcloud,4,2737,0.038253,0.096363
3,intel,bluehive,1,1347,0.002264,0.006687
4,intel,bluehive,2,2686,0.004681,0.017785
5,intel,bluehive,4,3999,0.004934,0.020884
6,intel,bluehive,8,2712,0.004852,0.207891
7,intel,gcloud,6,1351,0.007213,0.016820


In [128]:
test_results.groupby(["cpu_provider", "platform", "num_cores"]).apply(
    group_latency_metrics,
    include_groups=False
).reset_index()

,cpu_provider,platform,num_cores,count,rmse_log,within_10pct,within_25pct,within_50pct
0,amd,gcloud,1,2636.0,0.093501,0.889605,0.971927,0.992413
1,amd,gcloud,2,4015.0,0.143677,0.746700,0.948443,0.986800
2,amd,gcloud,4,2737.0,0.174169,0.678845,0.896967,0.973328
3,intel,bluehive,1,1347.0,0.206032,0.810690,0.933185,0.962880
4,intel,bluehive,2,2686.0,0.205916,0.846612,0.925540,0.960536
5,intel,bluehive,4,3999.0,0.231287,0.785446,0.889722,0.942986
6,intel,bluehive,8,2712.0,0.336973,0.656711,0.785398,0.897493
7,intel,gcloud,6,1351.0,0.229272,0.736491,0.874167,0.956329


In [129]:
cols_to_show = [
    "model",
    "cpu_provider",
    "platform",
    "num_cores",
    "true_latency_ms",
    "pred_latency_ms",
    "relative_error",
]

cols_to_show = [c for c in cols_to_show if c in test_results.columns]

In [130]:
test_results.sort_values("relative_error", ascending=True)[cols_to_show].head(5)

,model,cpu_provider,platform,num_cores,true_latency_ms,pred_latency_ms,relative_error
4195,gcresnext50ts_Opset18.onnx,intel,bluehive,2,37.606330,37.606328,4.637470e-08
6660,dla102x_Opset18_basic.onnx,intel,bluehive,8,29.713904,29.713838,2.216126e-06
18950,mt5_Opset16_disable_all.onnx,intel,bluehive,2,48.591429,48.591266,3.361537e-06
13905,lambda_resnet50ts_Opset16_basic.onnx,intel,bluehive,2,48.178016,48.177749,5.533924e-06
5969,resmlp_big_24_distilled_224_Opset17_disable_al...,amd,gcloud,4,675.973607,675.969456,6.140715e-06


In [131]:
test_results.sort_values("relative_error", ascending=False)[cols_to_show].head(5)

,model,cpu_provider,platform,num_cores,true_latency_ms,pred_latency_ms,relative_error
17722,gluon_resnext101_32x4d_Opset16.onnx,intel,bluehive,1,99.834912,642.823606,5.438866
18557,spnasnet_100_Opset18.onnx,amd,gcloud,4,2.735222,17.530404,5.409134
11202,fasterrcnn_mobilenet_v3_large_320_fpn_Opset17....,amd,gcloud,2,13.860660,82.575827,4.957568
3062,fbnetc_100_Opset18.onnx,amd,gcloud,4,3.089131,17.964602,4.815423
12608,spnasnet_100_Opset17.onnx,intel,gcloud,6,2.077749,11.954036,4.753359
